In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 32. 미니 LLM 파인튜닝 — LoRA + 양자화로 친절한 챗봇 [CPU/GPU 벤치마크 ⑭]

> **학습 목표**
> - 작은 오픈소스 모델(GPT-2 small 또는 작은 LLaMA)을 선택한다
> - QLoRA로 효율적으로 SFT를 수행한다
> - 양자화된 모델의 추론 속도를 CPU vs GPU로 비교한다

## 32.1 프로젝트 개요

Ch 31에서 만든 nano-GPT를 SFT + LoRA로 파인튜닝:
1. 베이스 모델: nano-GPT (사전학습 완료)
2. 미니 instruction 데이터셋 구성
3. LoRA 어댑터 학습
4. 양자화 추론 (CPU INT8 vs GPU FP16)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
import os

# Ch 31의 모델 정의 재사용 (간소화 버전)
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, theta_base=10000.0):
    B, H, T, d = x.shape
    freqs = 1.0 / (theta_base ** (torch.arange(0, d, 2).float() / d))
    angles = torch.arange(T, device=x.device).float()[:, None] * freqs[None, :]
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_q_heads
        self.n_rep = n_q_heads // n_kv_heads
        self.W_q = nn.Linear(d_model, n_q_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.d_k, d_model, bias=False)
    def forward(self, x, mask):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        q = apply_rope(q)
        k = apply_rope(k)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out)

class SwiGLUFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, d_ff):
        super().__init__()
        self.attn = GQAAttention(d_model, n_q_heads, n_kv_heads)
        self.ffn = SwiGLUFFN(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_q_heads=4, n_kv_heads=2,
                 n_layers=4, d_ff=512, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_q_heads, n_kv_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
    def forward(self, x):
        B, T = x.shape
        emb = self.token_emb(x) * (self.d_model ** 0.5)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        h = emb
        for block in self.blocks:
            h = block(h, mask)
        h = self.norm_f(h)
        return self.lm_head(h)

print("nano-GPT Model 정의 완료")


In [ ]:
# 데이터 준비 (Ch 31과 동일)
from llm_math.data import load_tiny_shakespeare

text = load_tiny_shakespeare(max_chars=50000)
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# 사전학습된 모델 로드 (있으면)
model = NanoGPT(vocab_size, d_model=128, n_q_heads=4, n_kv_heads=2,
                n_layers=4, d_ff=512, max_seq_len=256)

ckpt_path = '../checkpoints/nanogpt_shakespeare.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, weights_only=False)
    # strict=False로 로드 (이전 체크포인트에 LoRA Weight가 있을 수 있음)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print("Pretraining된 Model 로드 완료! (strict=False)")
else:
    print("체크포인트 없음. Ch 31을 먼저 실행하세요.")
    # 간단한 사전학습 (대안)
    print("간단한 Pretraining 수행 (50 스텝)...")
    data = np.array([char_to_idx[c] for c in text])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(50):
        starts = np.random.randint(0, len(data) - 129, 16)
        x = torch.tensor(np.stack([data[s:s+128] for s in starts]))
        y = torch.tensor(np.stack([data[s+1:s+129] for s in starts]))
        opt.zero_grad()
        loss = loss_fn(model(x).reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        opt.step()
    print(f"Pretraining 완료 (loss={loss.item():.4f})")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model 파라미터: {n_params:,}")


## 32.2 미니 Instruction 데이터셋


In [ ]:
# 미니 instruction 데이터 (Shakespeare 테마)
instructions = [
    {"instruction": "Write a line as Romeo.", "response": "O Romeo, Romeo! wherefore art thou Romeo?"},
    {"instruction": "Write a line as Juliet.", "response": "Parting is such sweet sorrow."},
    {"instruction": "Write a king's decree.", "response": "Let it be known throughout the land."},
    {"instruction": "Write a witch's prophecy.", "response": "All hail, Macbeth, that shalt be king hereafter!"},
    {"instruction": "Write Hamlet's soliloquy start.", "response": "To be, or not to be, that is the question."},
    {"instruction": "Write a fool's joke.", "response": "Better a witty fool than a foolish wit."},
    {"instruction": "Write a lover's plea.", "response": "If thou dost love, pronounce it faithfully."},
    {"instruction": "Write a ghost's warning.", "response": "The serpent that did sting thy father's life now wears his crown."},
]

def format_sft(ex):
    return f"### Instruction: {ex['instruction']}### Response: {ex['response']}<END>"

# 인코딩
def encode_sft(ex, max_len=128):
    text = format_sft(ex)
    ids = [char_to_idx[c] for c in text if c in char_to_idx]
    # 패딩
    while len(ids) < max_len:
        ids.append(0)
    return ids[:max_len]

def make_sft_batch(examples, max_len=128):
    inputs = [encode_sft(ex, max_len) for ex in examples]
    return torch.tensor(inputs, dtype=torch.long)

# Loss Mask (Response Subspace만 Training)
def make_response_mask(examples, max_len=128):
    masks = []
    for ex in examples:
        text = format_sft(ex)
        resp_start = text.find("### Response:") + len("### Response:")
        # Mask: response Subspace만 1
        mask = [0.0] * max_len
        for i in range(min(resp_start, max_len), min(len(text), max_len)):
            mask[i] = 1.0
        masks.append(mask)
    return torch.tensor(masks, dtype=torch.float32)

print(f"Instruction 예시: {instructions[0]}")
print(f"포맷: {format_sft(instructions[0])}")


## 32.3 LoRA 어댑터 추가


In [ ]:
# LoRA 어댑터
class LoRAAdapter(nn.Module):
    def __init__(self, d_in, d_out, r=4, alpha=8):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scaling = alpha / r
    def forward(self, x):
        # x: (..., d_in) -> (..., d_out)
        return (x @ self.A.t()) @ self.B.t() * self.scaling

# 모델의 어텐션 W_q, W_v에 LoRA 추가
def add_lora_to_model(model, r=4, alpha=8):
    lora_params = []
    for block in model.blocks:
        d_model = block.attn.W_q.in_features
        # W_q 출력: n_q_heads * d_k = d_model
        # W_v 출력: n_kv_heads * d_k
        q_out = block.attn.W_q.out_features
        v_out = block.attn.W_v.out_features
        block.attn.lora_q = LoRAAdapter(d_model, q_out, r=r, alpha=alpha)
        block.attn.lora_v = LoRAAdapter(d_model, v_out, r=r, alpha=alpha)
        lora_params.extend([block.attn.lora_q.A, block.attn.lora_q.B,
                           block.attn.lora_v.A, block.attn.lora_v.B])
    # 원래 파라미터 고정
    for p in model.parameters():
        p.requires_grad = False
    # LoRA만 학습
    for p in lora_params:
        p.requires_grad = True
    return lora_params

# 어텐션 forward 오버라이드
def attn_forward_with_lora(self, x, mask):
    B, T, D = x.shape
    q = self.W_q(x) + self.lora_q(x)
    k = self.W_k(x)
    v = self.W_v(x) + self.lora_v(x)
    q = q.view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
    k = k.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
    v = v.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
    q = apply_rope(q)
    k = apply_rope(k)
    if self.n_rep > 1:
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
    scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
    weights = F.softmax(scores, dim=-1)
    out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
    return self.W_o(out)

# LoRA 추가
lora_params = add_lora_to_model(model, r=4, alpha=8)
# forward 메서드 교체
for block in model.blocks:
    block.attn.forward = attn_forward_with_lora.__get__(block.attn, type(block.attn))

n_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"LoRA 추가 완료!")
print(f"  Trainable: {n_lora:,} ({n_lora/n_total*100:.2f}%)")
print(f"  Total: {n_total:,}")


## 32.4 SFT 학습


In [ ]:
# SFT 학습 (LoRA만)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)
loss_fn = nn.CrossEntropyLoss(reduction='none')

def sft_loss_with_mask(logits, targets, mask):
    B, T, V = logits.shape
    logits = logits[:, :-1, :].reshape(-1, V)
    targets = targets[:, 1:].reshape(-1)
    mask = mask[:, 1:].reshape(-1)
    losses = loss_fn(logits, targets)
    return (losses * mask).sum() / (mask.sum() + 1e-8)

# Training
n_epochs = 30
losses = []
for epoch in range(n_epochs):
    # 모든 Data 한 번씩 (Batch Magnitude 4)
    np.random.shuffle(instructions)
    total_loss = 0
    n_batches = 0
    for i in range(0, len(instructions), 4):
        batch = instructions[i:i+4]
        x = make_sft_batch(batch, max_len=128)
        mask = make_response_mask(batch, max_len=128)

        opt.zero_grad()
        logits = model(x)
        loss = sft_loss_with_mask(logits, x, mask)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        n_batches += 1
    losses.append(total_loss / n_batches)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: loss={losses[-1]:.4f}")

plt.figure(figsize=(9, 3))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('SFT Loss')
plt.title('LoRA SFT Learning Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch32_lora_sft.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# SFT 후 응답 생성
def generate_sft(model, instruction, n_new=100, temperature=0.7):
    model.eval()
    prompt = f"### Instruction: {instruction}### Response:"
    idx = [char_to_idx[c] for c in prompt if c in char_to_idx]
    with torch.no_grad():
        for _ in range(n_new):
            x = torch.tensor([idx[-256:]], dtype=torch.long)
            logits = model(x)
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()
            idx.append(next_idx)
            # <END> 감지
            text_so_far = ''.join(idx_to_char.get(i, '') for i in idx)
            if '<END>' in text_so_far:
                break
    return ''.join(idx_to_char.get(i, '') for i in idx)

print("=== SFT 후 Response (LoRA) ===\n")
for inst in ["Write a line as Romeo.", "Write a king's decree.", "Write Hamlet's soliloquy start."]:
    print(f"Q: {inst}")
    print(f"A: {generate_sft(model, inst, n_new=80, temperature=0.5)}")
    print()


## 32.5 양자화 추론


In [ ]:
# INT8 양자화 추론 (간소화)
import copy
def quantize_model_int8(model):
    """Model의 LinearLayer Weight를 INT8로 Quantization (복사본 반환)."""
    quantized_model = copy.deepcopy(model)
    for name, module in quantized_model.named_modules():
        if isinstance(module, nn.Linear):
            w = module.weight.data
            scale = w.abs().max() / 127.0
            q = torch.round(w / scale).clamp(-128, 127).to(torch.int8)
            # 역Quantization하여 저장 (실제 Inference 시 사용)
            module.weight.data = q.float() * scale
    return quantized_model

# FP32 vs INT8 Inference Speed Comparison
import time

# FP32 Inference
model.eval()
x = torch.randint(0, vocab_size, (1, 64))
def infer_fp32():
    with torch.no_grad():
        return model(x)

# warmup
infer_fp32()
t0 = time.perf_counter()
for _ in range(20):
    infer_fp32()
t_fp32 = (time.perf_counter() - t0) / 20 * 1000
print(f"FP32 추론: {t_fp32:.3f} ms")

# INT8 (시뮬레이션)
quantized = quantize_model_int8(model)
quantized.eval()
def infer_int8():
    with torch.no_grad():
        return quantized(x)

infer_int8()
t0 = time.perf_counter()
for _ in range(20):
    infer_int8()
t_int8 = (time.perf_counter() - t0) / 20 * 1000
print(f"INT8 추론 (시뮬레이션): {t_int8:.3f} ms")
print(f"Speed 비: {t_fp32/t_int8:.2f}x")
print("\n주의: 이것은 시뮬레이션. 실제 INT8 가속은 bitsandbytes, llama.cpp 등 필요.")


## 32.6 [CPU/GPU 벤치마크 ⑭] 양자화 추론 비교


In [ ]:
# 최종 벤치마크: CPU FP32 vs CPU INT8 vs GPU FP16
from llm_math.bench import time_fn

print("최종 Inference 벤치마크:")
print(f"{'Config':>20} | {'Time (ms)':>12} | {'Tokens/sec':>12}")
print("-" * 50)

# CPU FP32
def gen_one_token(model, x):
    with torch.no_grad():
        return model(x)
res_cpu_fp32 = time_fn(gen_one_token, model, x, device='cpu', warmup=3, repeat=10)
print(f"{'CPU FP32 (LoRA)':>20} | {res_cpu_fp32['mean_ms']:>12.3f} | {1000/res_cpu_fp32['mean_ms']:>12.1f}")

# CPU INT8 (시뮬레이션)
res_cpu_int8 = time_fn(gen_one_token, quantized, x, device='cpu', warmup=3, repeat=10)
print(f"{'CPU INT8 (sim)':>20} | {res_cpu_int8['mean_ms']:>12.3f} | {1000/res_cpu_int8['mean_ms']:>12.1f}")

if torch.cuda.is_available():
    model_gpu = model.cuda()
    x_gpu = x.cuda()
    res_gpu = time_fn(gen_one_token, model_gpu, x_gpu, device='cuda', warmup=3, repeat=10)
    print(f"{'GPU FP16 (LoRA)':>20} | {res_gpu['mean_ms']:>12.3f} | {1000/res_gpu['mean_ms']:>12.1f}")
else:
    print(f"{'GPU FP16':>20} | {'N/A':>12} | {'N/A':>12}")

print("\n=> 실제 INT8 Quantization는 bitsandbytes, llama.cpp 등으로 구현 시 2-3x 가속.")
print("=> GPU FP16은 CPU 대비 5-20x 가속 (Model Size에 따라).")


## 32.7 핵심 정리

이 장에서 수행한 것:
- ✅ 사전학습된 nano-GPT 로드
- ✅ 미니 instruction 데이터셋 구성
- ✅ LoRA 어댑터 추가 (Q, V에 r=4)
- ✅ 손실 마스킹 SFT 학습
- ✅ INT8 양자화 시뮬레이션
- ✅ CPU vs GPU 추론 벤치마크

## 32.8 한계와 개선 방향

한계:
- 모델이 너무 작아 (1M 파라미터) 의미 있는 챗봇 불가
- 문자 단위 토크나이저 (실제는 BPE)
- INT8 양자화가 시뮬레이션 (실제 가속 아님)

개선:
- GPT-2 small (124M) 이상으로 확장
- HuggingFace transformers + datasets 사용
- bitsandbytes로 실제 4-bit 양자화
- 더 큰 instruction 데이터셋 (Alpaca, OpenAssistant)

## 연습문제

1. LoRA rank r=2, 4, 8, 16으로 학습하고 성능을 비교하라.
2. LoRA를 QKV 전부에 적용 vs Q, V만 적용을 비교하라.
3. 더 많은 instruction 데이터(50, 100개)로 학습하고 응답 품질을 비교하라.
4. 양자화 전후 PPL을 비교하라.
5. HuggingFace transformers로 GPT-2를 로드하고 같은 파인튜닝을 수행하라.

> 해설: `solutions/ch32_solutions.ipynb`
